# Ego network — u/CaseA

Directed ego network built from the [Arctic Shift](https://arctic-shift.photon-reddit.com) API
(Pushshift successor, no credentials required).

**Nodes** — the ego plus every account the ego replied to directly, and every account that
replied to the ego directly.

**Edges** — directed. `ego -> alter` means the ego replied to that alter's content;
`alter -> ego` means the alter replied to the ego's content.

**Weights** — frequency of interaction, discounted for repetition. Each *distinct post*
on which a pair interacts contributes a full `1.0`. Repeat interactions on a post the pair
is *already* interacting on contribute `1 - exp(-dt / tau)`, where `dt` is the time since
that pair's previous interaction in that direction on that same post. A rapid back-and-forth
in one comment thread therefore adds almost nothing beyond the first exchange, while
interacting across several different posts accumulates full weight each time.

Section 7 verifies these properties hold, on synthetic fixtures with known answers and on
the collected data.

---
## API notes verified against the live endpoint before writing this notebook

| Finding | Consequence |
|---|---|
| `meta=true` is rejected (`Unknown query parameter: 'meta'`) | not sent; `_meta` is not needed here |
| `parent_id` search accepts **only** `t1_` (comment) parents | replies to the ego's *submissions* must be found via `link_id` + filter, **not** `parent_id` |
| `parent_id=<bare base36>` returns `200` with `0` rows | a silent wrong answer — always send the `t1_` prefix |
| `/ids?ids=` silently truncates above 100 ids | batch at 100 and assert the returned count |
| `limit` is capped at 100 | pagination via the `after` cursor |
| default Python `urllib` UA gets `403` | an explicit `User-Agent` is set on the session |


## 1. Imports and configuration

In [ ]:
#import libraries
import pandas as pd
import numpy as np
import os
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)
import re
import json
import time
import math
import requests
from requests_ratelimiter import LimiterSession
import pyarrow as pa
import pyarrow.parquet as pq
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [ ]:
# ---- configuration -------------------------------------------------------
EGO = 'CaseA'          # canonical casing as returned by the API

# Decay constant for repeat interactions on a post the pair already interacts on.
# w = 1 - exp(-dt / TAU). At dt = TAU the repeat is worth ~0.63; well below TAU it
# is worth ~0. Section 7.3 sweeps this value.
TAU_SECONDS = 6 * 3600              # 6 hours

# A top-level comment on a submission is treated as a direct reply to the
# submission's author (and vice versa). Set False to keep comment->comment only.
INCLUDE_SUBMISSION_LEVEL = True

# Placeholder authors are not accounts and must never become nodes.
NON_AUTHORS = {'[deleted]', '[removed]', None, ''}
# The ego is excluded as an alter so self-replies cannot create a self-loop.

# Derived data stays beside this notebook; figures go to the project's figure
# directory (see CLAUDE.md layout). Parquet + zstd is the project's default format.
OUT_DIR = 'ego_network_output'
FIG_DIR = os.path.join('..', 'p2_figures')
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print(f'ego = {EGO}  tau = {TAU_SECONDS}s ({TAU_SECONDS/3600:.1f}h)  '
      f'submission-level = {INCLUDE_SUBMISSION_LEVEL}')

## 2. API layer

One rate-limited session for the whole notebook (a session created per call resets the
limiter and gets the collection 429'd). Every network loop carries the project's
`retry_count < 10` guard and re-raises once exhausted.

In [ ]:
# Arctic Shift API helpers: one rate-limited session (5 requests/second), a GET with retries, a paginator, and id lookups.
BASE = 'https://arctic-shift.photon-reddit.com/api'

session = LimiterSession(per_second=5)
session.headers['User-Agent'] = 'dissertation-ego-network/1.0 (research)'


def arctic_get(path, **params):
    """One GET with the standard retry guard. Raises after 10 failed attempts."""
    retry_count = 0
    last_error = None
    while retry_count < 10:
        try:
            response = session.get(f'{BASE}/{path}', params=params, timeout=60)
            if response.status_code == 200:
                return response.json().get('data') or []
            # 400 means a malformed query - retrying cannot fix it.
            if response.status_code == 400:
                raise ValueError(
                    f'400 from {path} {params}: {response.json().get("error")}'
                )
            last_error = f'HTTP {response.status_code}'
            print(f'API response: {response.status_code} for {path} {params}')
            retry_count += 1
            time.sleep(2 ** retry_count * 0.1)
        except ValueError:
            raise
        except Exception as e:
            last_error = f'{e.__class__.__name__}: {e}'
            print(f'Error for {path} {params}: {last_error}')
            retry_count += 1
            time.sleep(2 ** retry_count * 0.1)
    raise RuntimeError(f'Failed {path} {params} after 10 retries. Last: {last_error}')


def arctic_paginate(path, **params):
    """Page a search endpoint with the ascending created_utc cursor.

    Breaks on a short page (len < 100), never on len == 100. Deduplicates on `id`
    because the `after` cursor is inclusive at the boundary on some endpoints.
    """
    rows, seen = [], set()
    after = None
    while True:
        page_params = dict(params, limit=100, sort='asc')
        if after is not None:
            page_params['after'] = after
        data = arctic_get(path, **page_params)
        for record in data:
            if record['id'] not in seen:
                seen.add(record['id'])
                rows.append(record)
        if len(data) < 100:
            break
        next_after = data[-1]['created_utc']
        if next_after == after:          # cursor stuck - stop rather than spin
            print(f'cursor stalled at {after} for {path} {params}')
            break
        after = next_after
    return rows


def fetch_by_ids(kind, ids):
    """Look up `posts` or `comments` by id. Batches at 100 - the endpoint returns
    200 and silently truncates above that."""
    ids = [i for i in dict.fromkeys(ids) if i]
    out = []
    for start in range(0, len(ids), 100):
        batch = ids[start:start + 100]
        data = arctic_get(f'{kind}/ids', ids=','.join(batch))
        assert len(data) <= len(batch), (
            f'{kind}/ids returned {len(data)} for {len(batch)} requested ids'
        )
        out.extend(data)
    return out


print('API layer ready')

## 3. Collect the ego's own activity

Everything downstream is seeded from these two pulls, so they are checked first.

In [ ]:
# Every comment and post the ego account made.
ego_comments = arctic_paginate('comments/search', author=EGO)
ego_posts    = arctic_paginate('posts/search',    author=EGO)

print(f'ego comments: {len(ego_comments)}')
print(f'ego posts   : {len(ego_posts)}')

In [ ]:
# --- checks on the seed pulls ---
assert ego_comments or ego_posts, f'no activity found for {EGO} - check the spelling'

# The API matches author case-insensitively; confirm every row really is the ego.
authors = {c['author'] for c in ego_comments} | {p['author'] for p in ego_posts}
assert authors <= {EGO}, f'unexpected authors in the ego pull: {authors - {EGO}}'

# Pagination must not have duplicated or lost rows.
assert len({c['id'] for c in ego_comments}) == len(ego_comments), 'duplicate comment ids'
assert len({p['id'] for p in ego_posts})    == len(ego_posts),    'duplicate post ids'

_ts = [c['created_utc'] for c in ego_comments] + [p['created_utc'] for p in ego_posts]
print(f'PASS  {len(ego_comments)} comments + {len(ego_posts)} posts, all authored by {EGO}')
print(f'      activity spans {pd.to_datetime(min(_ts), unit="s")} '
      f'-> {pd.to_datetime(max(_ts), unit="s")}')
print(f'      subreddits: '
      f'{sorted({c["subreddit"] for c in ego_comments} | {p["subreddit"] for p in ego_posts})}')

## 4. Outgoing edges — who the ego replied to

Every comment the ego wrote has a `parent_id`:

- `t1_<id>` — a reply to another **comment**; the alter is that comment's author.
- `t3_<id>` — a top-level comment on a **submission**; the alter is the submission's author
  (included only when `INCLUDE_SUBMISSION_LEVEL` is set).

Parents are resolved in bulk through the `/ids` endpoints rather than assumed.

In [ ]:
# Look up the author of whatever each ego comment replied to (by id); report parents missing from the archive.
# The next cell turns these into outgoing ties.
parent_comment_ids = [c['parent_id'][3:] for c in ego_comments
                      if c['parent_id'].startswith('t1_')]
parent_post_ids     = [c['parent_id'][3:] for c in ego_comments
                      if c['parent_id'].startswith('t3_')]

print(f'ego comments replying to a comment   : {len(parent_comment_ids)}')
print(f'ego comments replying to a submission: {len(parent_post_ids)}')

parent_comments = fetch_by_ids('comments', parent_comment_ids)
parent_posts    = fetch_by_ids('posts',    parent_post_ids)

# id -> author lookups
comment_author = {c['id']: c['author'] for c in parent_comments}
post_author    = {p['id']: p['author'] for p in parent_posts}

unresolved = ([i for i in set(parent_comment_ids) if i not in comment_author]
              + [i for i in set(parent_post_ids) if i not in post_author])
if unresolved:
    print(f'WARNING  {len(unresolved)} parent(s) not in the archive, edge(s) dropped: '
          f'{unresolved}')
else:
    print('PASS  every parent resolved to an author')

In [ ]:
outgoing = []
skipped_out = {'unresolved': 0, 'non_author': 0, 'self': 0, 'submission_level': 0}

for c in ego_comments:
    ptype, pid = c['parent_id'][:3], c['parent_id'][3:]

    if ptype == 't1_':
        alter = comment_author.get(pid)
        kind = 'comment_reply'
    elif ptype == 't3_':
        if not INCLUDE_SUBMISSION_LEVEL:
            skipped_out['submission_level'] += 1
            continue
        alter = post_author.get(pid)
        kind = 'submission_reply'
    else:
        raise ValueError(f'unexpected parent_id prefix: {c["parent_id"]}')

    if alter is None:
        skipped_out['unresolved'] += 1
        continue
    if alter in NON_AUTHORS:
        skipped_out['non_author'] += 1
        continue
    if alter == EGO:
        skipped_out['self'] += 1
        continue

    outgoing.append({
        'source': EGO,
        'target': alter,
        'direction': 'out',
        'kind': kind,
        # link_id is the submission the interaction happened on - the unit the
        # repetition discount is grouped by.
        'link_id': c['link_id'],
        'created_utc': c['created_utc'],
        'interaction_id': c['id'],       # the comment that constitutes the reply
        'subreddit': c['subreddit'],
    })

print(f'outgoing interactions: {len(outgoing)}')
print(f'skipped: {skipped_out}')
print(f'distinct alters replied to: {len({e["target"] for e in outgoing})}')

## 5. Incoming edges — who replied to the ego

Two different queries are needed, and this is the notebook's main correctness hazard:

1. **Replies to the ego's comments** — `comments/search?parent_id=t1_<comment_id>`.
2. **Replies to the ego's submissions** — `parent_id` **cannot** be used. The endpoint
   rejects a `t3_` value outright, and a bare base36 id is silently read as a *comment*
   id, returning `200` with zero rows. The correct route is to pull every comment on the
   submission via `link_id` and keep those whose `parent_id` equals `t3_<post_id>`.

The next cell demonstrates that failure mode rather than taking it on trust.

In [ ]:
# --- demonstrate why parent_id cannot be used for submissions ---
if ego_posts:
    probe = ego_posts[0]['id']
    try:
        arctic_get('comments/search', parent_id=f't3_{probe}', limit=100)
        print('NOTE  t3_ parent_id was accepted - the API changed; re-check section 5')
    except ValueError as e:
        print(f'as expected, t3_ prefix is rejected:\n  {e}\n')

    bare = arctic_get('comments/search', parent_id=probe, limit=100)
    print(f'bare id parent_id={probe} -> HTTP 200 with {len(bare)} rows '
          f'(silently wrong; the post has {ego_posts[0]["num_comments"]} comments)')
    print('=> submissions are handled via link_id + filter below.')

In [ ]:
# Incoming ties: replies by other accounts to the ego (self-replies and deleted authors skipped).
incoming = []
self_replies = 0

# --- 5a. replies to the ego's comments -----------------------------------
for c in ego_comments:
    replies = arctic_paginate('comments/search', parent_id=f't1_{c["id"]}')
    for r in replies:
        assert r['parent_id'] == f't1_{c["id"]}', (
            f'parent_id search returned a non-child: {r["parent_id"]}'
        )
        if r['author'] == EGO:
            self_replies += 1
            continue
        if r['author'] in NON_AUTHORS:
            continue
        incoming.append({
            'source': r['author'],
            'target': EGO,
            'direction': 'in',
            'kind': 'comment_reply',
            'link_id': r['link_id'],
            'created_utc': r['created_utc'],
            'interaction_id': r['id'],
            'subreddit': r['subreddit'],
        })

print(f'replies to ego comments: {len(incoming)}  (self-replies skipped: {self_replies})')

In [ ]:
# --- 5b. top-level comments on the ego's submissions ----------------------
submission_thread_cache = {}     # post_id -> every archived comment on it
n_before = len(incoming)

if INCLUDE_SUBMISSION_LEVEL:
    for p in ego_posts:
        thread = arctic_paginate('comments/search', link_id=f't3_{p["id"]}')
        submission_thread_cache[p['id']] = thread
        top_level = [r for r in thread if r['parent_id'] == f't3_{p["id"]}']
        print(f'  post {p["id"]} r/{p["subreddit"]}: '
              f'{len(thread)} comments archived, {len(top_level)} top-level '
              f'(num_comments snapshot: {p["num_comments"]})')
        for r in top_level:
            if r['author'] == EGO:
                self_replies += 1
                continue
            if r['author'] in NON_AUTHORS:
                continue
            incoming.append({
                'source': r['author'],
                'target': EGO,
                'direction': 'in',
                'kind': 'submission_reply',
                'link_id': f't3_{p["id"]}',
                'created_utc': r['created_utc'],
                'interaction_id': r['id'],
                'subreddit': r['subreddit'],
            })

print(f'\ntop-level replies to ego submissions: {len(incoming) - n_before}')
print(f'incoming interactions total: {len(incoming)}')
print(f'distinct alters who replied: {len({e["source"] for e in incoming})}')

## 6. The interaction table

One row per directed interaction. This is the input to the weighting function.

In [ ]:
interactions = pd.DataFrame(outgoing + incoming)
interactions = interactions.sort_values('created_utc').reset_index(drop=True)
interactions['datetime'] = pd.to_datetime(interactions['created_utc'], unit='s')

# One reddit comment must never appear as two interactions.
assert interactions['interaction_id'].is_unique, 'duplicate interaction_id rows'

# Every interaction must involve the ego on exactly one side - this is an ego network.
touches_ego = (interactions['source'] == EGO) ^ (interactions['target'] == EGO)
assert touches_ego.all(), 'found interactions that are not incident to the ego'

print(f'{len(interactions)} interactions')
print(interactions['direction'].value_counts().to_string())
print()
print(interactions.groupby(['direction', 'kind']).size().to_string())
interactions.head(10)

## 7. Edge weights

For one directed pair `(source, target)`:

1. Group that pair's interactions by `link_id` (the submission they happened on).
2. Sort each group by time. The **first** interaction on a post scores `1.0`.
3. Each later interaction on that *same* post scores `1 - exp(-dt / tau)`, `dt` being the
   gap to that pair's previous interaction in that direction on that post.
4. The edge weight is the sum across all posts.

So the weight is approximately *the number of distinct posts the pair interacted on*,
plus a small fraction for within-post repeats — which is the requested ordering: several
different posts outrank a rapid back-and-forth on one.

In [ ]:
# Edge weights with a repetition discount (TAU_SECONDS): a repeat on the same post counts 1 - exp(-gap/tau).
def compute_edge_weights(df, tau=TAU_SECONDS, return_detail=False):
    """Weight each directed edge by discounted interaction frequency.

    First interaction on a given post scores 1.0; each repeat on that same post
    scores 1 - exp(-dt / tau) against the previous interaction for that edge on
    that post. Weight = sum over all of the edge's posts.
    """
    rows, detail = [], []
    if len(df) == 0:
        empty = pd.DataFrame(columns=['source', 'target', 'weight', 'n_interactions',
                                      'n_posts', 'direction'])
        return (empty, pd.DataFrame()) if return_detail else empty

    for (src, tgt), pair in df.groupby(['source', 'target'], sort=False):
        weight = 0.0
        for link_id, grp in pair.groupby('link_id', sort=False):
            times = np.sort(grp['created_utc'].to_numpy())
            weight += 1.0                                   # first contact on this post
            if len(times) > 1:
                gaps = np.diff(times)                       # >= 0 after sorting
                contrib = 1.0 - np.exp(-gaps / tau)
                weight += contrib.sum()
                if return_detail:
                    for gap, w in zip(gaps, contrib):
                        detail.append({'source': src, 'target': tgt, 'link_id': link_id,
                                       'gap_seconds': int(gap), 'repeat_weight': float(w)})
        rows.append({
            'source': src,
            'target': tgt,
            'weight': weight,
            'n_interactions': len(pair),
            'n_posts': pair['link_id'].nunique(),
            'direction': pair['direction'].iloc[0],
        })

    edges = pd.DataFrame(rows).sort_values('weight', ascending=False).reset_index(drop=True)
    return (edges, pd.DataFrame(detail)) if return_detail else edges


# Weights for this ego network, keeping the per-repeat detail.
edges, repeat_detail = compute_edge_weights(interactions, return_detail=True)
print(f'{len(edges)} directed edges')
edges.head(15)

### 7.1 Property tests on synthetic fixtures

These do not touch the API. Each fixture has a hand-computed expected answer, so the
weighting is verified against the stated intent rather than inspected for plausibility.

In [ ]:
# Property tests (T1-T8) for compute_edge_weights on small synthetic inputs.
def _fixture(pairs, source='EGO_T', target='ALTER_T'):
    """pairs: list of (link_id, created_utc)."""
    return pd.DataFrame([
        {'source': source, 'target': target, 'direction': 'out', 'kind': 'comment_reply',
         'link_id': lid, 'created_utc': t, 'interaction_id': f'x{i}', 'subreddit': 's'}
        for i, (lid, t) in enumerate(pairs)
    ])

T0 = 1_600_000_000
TAU = TAU_SECONDS
failures = []

def check(name, condition, detail=''):
    status = 'PASS' if condition else 'FAIL'
    if not condition:
        failures.append(name)
    print(f'  [{status}] {name}{"  " + detail if detail else ""}')


print('T1  one interaction on each of N distinct posts -> weight == N')
for n in (1, 3, 8):
    w = compute_edge_weights(
        _fixture([(f't3_p{i}', T0 + i * 86400) for i in range(n)]), TAU
    )['weight'].iloc[0]
    check(f'n_posts={n} -> weight {w:.4f}', abs(w - n) < 1e-9, f'(expected {n})')

print()
print('T2  N rapid interactions on ONE post -> weight ~ 1, far below N')
burst = _fixture([('t3_p0', T0 + i * 60) for i in range(8)])   # 8 replies, 1 min apart
w_burst = compute_edge_weights(burst, TAU)['weight'].iloc[0]
check(f'8 replies 60s apart -> weight {w_burst:.4f}', w_burst < 1.2,
      '(expected just above 1.0)')

print()
print('T3  the ordering the brief asks for: same interaction count, '
      'multi-post beats one-post burst')
w_multi = compute_edge_weights(
    _fixture([(f't3_p{i}', T0 + i * 86400) for i in range(8)]), TAU
)['weight'].iloc[0]
check(f'8 posts ({w_multi:.4f}) > 8 replies on 1 post ({w_burst:.4f})',
      w_multi > w_burst, f'ratio {w_multi / w_burst:.2f}x')

print()
print('T4  a back-and-forth thread is worth barely more than a single reply')
w_single = compute_edge_weights(_fixture([('t3_p0', T0)]), TAU)['weight'].iloc[0]
check(f'single reply {w_single:.4f} vs back-and-forth {w_burst:.4f}',
      w_burst - w_single < 0.25, '(burst adds < 0.25 over one reply)')

print()
print('T5  monotonic in the gap: the longer the wait, the more a repeat counts')
gap_weights = [
    compute_edge_weights(_fixture([('t3_p0', T0), ('t3_p0', T0 + g)]), TAU)['weight'].iloc[0]
    for g in (60, 3600, 6 * 3600, 86400, 30 * 86400)
]
check('weight strictly increases with gap',
      all(a < b for a, b in zip(gap_weights, gap_weights[1:])),
      '  ' + ', '.join(f'{g}: {w:.4f}' for g, w in
                       zip(['1m', '1h', '6h', '1d', '30d'], gap_weights)))
check('a 30-day repeat on the same post approaches a fresh post (2.0)',
      abs(gap_weights[-1] - 2.0) < 0.01, f'({gap_weights[-1]:.4f})')

print()
print('T6  bounds: 1 <= weight <= n_interactions, always')
rng = np.random.default_rng(0)
for trial in range(200):
    n = int(rng.integers(1, 12))
    fx = _fixture([(f't3_p{rng.integers(0, 4)}', T0 + int(rng.integers(0, 90 * 86400)))
                   for _ in range(n)])
    r = compute_edge_weights(fx, TAU).iloc[0]
    if not (1.0 - 1e-9 <= r['weight'] <= n + 1e-9):
        check(f'bounds violated on trial {trial}', False, f'{r["weight"]} vs n={n}')
        break
else:
    check('200 randomised fixtures all satisfy 1 <= weight <= n', True)

print()
print('T7  simultaneous interactions (dt = 0) add nothing and do not error')
w_same = compute_edge_weights(_fixture([('t3_p0', T0), ('t3_p0', T0)]), TAU)['weight'].iloc[0]
check(f'two interactions at the same second -> {w_same:.4f}', abs(w_same - 1.0) < 1e-9)

print()
print('T8  unsorted input gives the same answer as sorted input')
shuffled = _fixture([('t3_p0', T0 + 5000), ('t3_p0', T0), ('t3_p1', T0 + 200)])
w_shuf = compute_edge_weights(shuffled.sample(frac=1, random_state=3), TAU)['weight'].iloc[0]
w_sort = compute_edge_weights(shuffled.sort_values('created_utc'), TAU)['weight'].iloc[0]
check(f'shuffled {w_shuf:.6f} == sorted {w_sort:.6f}', abs(w_shuf - w_sort) < 1e-12)

print()
print('=' * 68)
print('ALL PROPERTY TESTS PASSED' if not failures else f'FAILURES: {failures}')
assert not failures, failures

### 7.2 Checks against the collected data

The synthetic tests confirm the function is correct; these confirm it was applied to the
right rows, and that the graph really is a directed ego network.

In [ ]:
checks = []
def dcheck(name, condition, detail=''):
    checks.append((name, bool(condition)))
    print(f'  [{"PASS" if condition else "FAIL"}] {name}{"  " + detail if detail else ""}')

# --- structure ---
dcheck('every edge is incident to the ego',
       ((edges['source'] == EGO) ^ (edges['target'] == EGO)).all())
dcheck('no self-loops', not ((edges['source'] == edges['target']).any()))
dcheck('no placeholder accounts as nodes',
       not (set(edges['source']) | set(edges['target'])) & set(NON_AUTHORS))
dcheck("direction column matches the ego's position",
       ((edges['direction'] == 'out') == (edges['source'] == EGO)).all())
dcheck('one row per ordered pair',
       not edges.duplicated(subset=['source', 'target']).any())

# --- conservation: no interaction lost or double counted ---
dcheck('interaction counts sum back to the interaction table',
       edges['n_interactions'].sum() == len(interactions),
       f'{edges["n_interactions"].sum()} vs {len(interactions)}')

# --- weights ---
dcheck('all weights >= 1', (edges['weight'] >= 1 - 1e-9).all())
dcheck('no weight exceeds its interaction count',
       (edges['weight'] <= edges['n_interactions'] + 1e-9).all())
dcheck('no weight is below its distinct-post count',
       (edges['weight'] >= edges['n_posts'] - 1e-9).all())
dcheck('weight equals n_posts exactly where there are no repeats',
       np.allclose(edges.loc[edges['n_interactions'] == edges['n_posts'], 'weight'],
                   edges.loc[edges['n_interactions'] == edges['n_posts'], 'n_posts']))

# --- the discount actually discounts ---
repeated = edges[edges['n_interactions'] > edges['n_posts']]
if len(repeated):
    dcheck('every edge with within-post repeats is weighted below its raw count',
           (repeated['weight'] < repeated['n_interactions'] - 1e-9).all(),
           f'{len(repeated)} such edge(s)')
else:
    print('  [note] no pair repeated within a single post, so the discount never '
          'fires on this data; T2/T3/T4 above prove it fires when it should')

print()
print('=' * 68)
print('ALL DATA CHECKS PASSED' if all(c for _, c in checks)
      else f'FAILURES: {[n for n, c in checks if not c]}')
assert all(c for _, c in checks)

### 7.3 Sensitivity to `tau`

`tau` is the one free parameter. If the edge ranking moved with it, the conclusions would
be an artefact of the choice rather than a property of the data. The sweep is compared
against two reference schemes: raw interaction counts (no discount at all) and distinct-post
counts (a total discount).

`tau` can only ever change a weight for a pair that interacted **more than once on the same
post**. If no pair did, every weight equals its distinct-post count and the sweep is
degenerate by construction — the cell detects that case, says so, and demonstrates the
`tau` response on a synthetic pair instead, so the section still shows what the parameter
does.

In [ ]:
# Sensitivity of edge weights to tau (10 min to 7 days). If no pair repeats on a post, tau cannot matter;
# then a synthetic example shows the effect instead.
taus = [600, 3600, 6 * 3600, 86400, 7 * 86400]
tau_label = lambda t: f'tau={t//3600}h' if t >= 3600 else f'tau={t//60}m'

sens = edges[['source', 'target', 'n_interactions', 'n_posts']].copy()
for t in taus:
    w = compute_edge_weights(interactions, tau=t).set_index(['source', 'target'])['weight']
    sens[tau_label(t)] = pd.MultiIndex.from_frame(sens[['source', 'target']]).map(w)
rank_cols = [tau_label(t) for t in taus]

n_repeat_edges = int((edges['n_interactions'] > edges['n_posts']).sum())

if n_repeat_edges == 0:
    print('DEGENERATE: no pair interacted more than once on the same post '
          f'({n_repeat_edges} such edges), so tau cannot affect any weight here.')
    spread = sens[rank_cols].max(axis=1) - sens[rank_cols].min(axis=1)
    print(f'  max weight change across tau = {spread.max():.2e} (identical, as expected)')
    assert np.allclose(spread, 0), 'tau changed a weight despite no within-post repeats'
    assert np.allclose(sens[rank_cols[0]], sens['n_posts']), \
        'with no repeats every weight must equal the distinct-post count'
    print('  PASS  every weight equals its distinct-post count at every tau')
    print()
    print('Synthetic demonstration of the tau response '
          '(3 interactions on ONE post, 30 min apart):')
    demo = _fixture([('t3_p0', T0), ('t3_p0', T0 + 1800), ('t3_p0', T0 + 3600)])
    for t in taus:
        w = compute_edge_weights(demo, tau=t)['weight'].iloc[0]
        print(f'  {tau_label(t):>9}  weight {w:.4f}   '
              f'(raw count 3, distinct-post count 1)')
    print('  -> a shorter tau discounts the burst harder; all settings stay well '
          'below the raw count of 3.')
else:
    print(f'{n_repeat_edges} edge(s) have within-post repeats, so tau is live.')
    print('Spearman rank correlation of edge weights between tau settings:')
    print(sens[rank_cols].corr(method='spearman').round(4).to_string())
    print()
    print('Correlation with the two reference schemes:')
    for c in rank_cols:
        print(f'  {c:>10}  vs raw interaction count: '
              f'{sens[c].corr(sens["n_interactions"], method="spearman"):.4f}'
              f'   vs distinct-post count: '
              f'{sens[c].corr(sens["n_posts"], method="spearman"):.4f}')

sens.head(10)

### 7.4 Spot-check against reddit itself

The top edges are printed with their permalinks so a handful can be opened and confirmed
by eye — the final check that the graph describes real interactions.

In [ ]:
# The five heaviest edges and their first four interactions, with links, for a hand check.
top = edges.head(5)
for _, e in top.iterrows():
    rows = interactions[(interactions['source'] == e['source'])
                        & (interactions['target'] == e['target'])]
    arrow = f'{e["source"]} -> {e["target"]}'
    print(f'{arrow}   weight {e["weight"]:.4f}  '
          f'({e["n_interactions"]} interaction(s) across {e["n_posts"]} post(s))')
    for _, r in rows.sort_values('created_utc').head(4).iterrows():
        print(f'    {r["datetime"]}  r/{r["subreddit"]}  {r["kind"]}  '
              f'https://reddit.com/comments/{r["link_id"][3:]}//{r["interaction_id"]}')
    print()

## 8. Build the directed graph

In [ ]:
# Directed ego network from the weighted edges; checks it is a star on the ego. Reciprocated alters = edges both ways.
G = nx.DiGraph()
G.add_node(EGO, role='ego')
for _, e in edges.iterrows():
    for n in (e['source'], e['target']):
        if n != EGO and n not in G:
            G.add_node(n, role='alter')
    G.add_edge(e['source'], e['target'],
               weight=float(e['weight']),
               n_interactions=int(e['n_interactions']),
               n_posts=int(e['n_posts']),
               direction=e['direction'])

# The ego network must be a star on the ego: every node adjacent to it, no other edges.
assert G.number_of_edges() == len(edges)
assert all(EGO in (u, v) for u, v in G.edges()), 'edge not incident to the ego'
assert set(G.nodes()) - {EGO} == set(nx.all_neighbors(G, EGO)) - {EGO}

alters = set(G.nodes()) - {EGO}
mutual = {n for n in alters if G.has_edge(EGO, n) and G.has_edge(n, EGO)}

print(f'nodes: {G.number_of_nodes()}  ({len(alters)} alters + ego)')
print(f'edges: {G.number_of_edges()}')
print(f'  ego -> alter: {G.out_degree(EGO)}')
print(f'  alter -> ego: {G.in_degree(EGO)}')
print(f'reciprocated (both directions): {len(mutual)}  '
      f'{sorted(mutual)[:10] if mutual else ""}')
print(f'total out-weight from ego: {G.out_degree(EGO, weight="weight"):.3f}')
print(f'total in-weight to ego   : {G.in_degree(EGO, weight="weight"):.3f}')

## 9. Visualisation

Radial layout with the ego at the centre. Alters are ordered around the ring by their
relationship to the ego — reciprocated first, then the ones the ego replied to, then the
ones who only replied in — so direction reads as a contiguous block rather than scattered
arrows. Colour carries direction (a two-hue categorical pair validated for colour-vision
deficiency); width carries weight, and the subtitle says so explicitly when every weight is
tied and width therefore carries nothing.

Labels go to every alter the ego replied to, then to an evenly spaced sample of the rest —
labelling all 235 would be unreadable, and picking "the strongest" is meaningless when the
weights are tied. The full ranking is the table in section 10.

In [ ]:
EGO_COLOR = '#0b0b0b'
OUT_COLOR = '#eb6834'   # ego -> alter
IN_COLOR  = '#2a78d6'   # alter -> ego
MUTED     = '#52514e'
SURFACE   = '#fcfcfb'

def draw_ego_network(G, ego, max_labels=16, figsize=(12, 12), title=None, subtitle=None):
    alters = [n for n in G.nodes() if n != ego]
    w_out = {n: (G[ego][n]['weight'] if G.has_edge(ego, n) else 0.0) for n in alters}
    w_in  = {n: (G[n][ego]['weight'] if G.has_edge(n, ego) else 0.0) for n in alters}
    strength = {n: w_out[n] + w_in[n] for n in alters}

    def group(n):
        if w_out[n] and w_in[n]:
            return 0           # reciprocated
        return 1 if w_out[n] else 2   # ego -> alter only, then alter -> ego only
    ordered = sorted(alters, key=lambda n: (group(n), -strength[n], n))

    n = len(ordered)
    two_rings = n > 60          # stagger radii when crowded so nodes stay distinct
    pos = {ego: np.array([0.0, 0.0])}
    ang_of = {}
    for i, node in enumerate(ordered):
        a = 2 * np.pi * i / max(n, 1) - np.pi / 2
        r = 1.0 if (not two_rings or i % 2 == 0) else 0.78
        ang_of[node] = a
        pos[node] = np.array([r * np.cos(a), r * np.sin(a)])

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_facecolor(SURFACE); fig.patch.set_facecolor(SURFACE)

    weights = [d['weight'] for _, _, d in G.edges(data=True)]
    wmin, wmax = min(weights), max(weights)
    tied = (wmax - wmin) < 1e-9
    lw = lambda w: 1.0 if tied else 0.6 + 3.4 * (w - wmin) / (wmax - wmin)

    for u, v, d in G.edges(data=True):
        out = (u == ego)
        ax.annotate('', xy=pos[v], xytext=pos[u],
                    arrowprops=dict(arrowstyle='-|>', mutation_scale=8,
                                    color=OUT_COLOR if out else IN_COLOR,
                                    lw=lw(d['weight']), alpha=0.85 if out else 0.45,
                                    shrinkA=7, shrinkB=7,
                                    connectionstyle=f'arc3,rad={0.06 if out else -0.06}'))

    smin, smax = min(strength.values()), max(strength.values())
    sizes = [34 + 130 * ((strength[nd] - smin) / max(smax - smin, 1e-9)) for nd in ordered]
    ax.scatter([pos[nd][0] for nd in ordered], [pos[nd][1] for nd in ordered],
               s=sizes, c=[OUT_COLOR if w_out[nd] else '#ffffff' for nd in ordered],
               edgecolors=MUTED, linewidths=0.8, zorder=3)
    ax.scatter([0], [0], s=420, c=EGO_COLOR, zorder=4)
    # Label sits below the node rather than inside it: a long username overflows the
    # marker and white-on-surface text then vanishes.
    ax.text(0, -0.075, f'u/{ego}', ha='center', va='top', color=EGO_COLOR,
            fontsize=9, fontweight='bold', zorder=5,
            bbox=dict(boxstyle='round,pad=0.22', fc=SURFACE, ec='none', alpha=0.9))

    # label everyone the ego replied to, then an evenly spaced sample of the rest
    to_label = [nd for nd in ordered if w_out[nd]]
    rest = [nd for nd in ordered if not w_out[nd]]
    slots = max(max_labels - len(to_label), 0)
    if slots and rest:
        to_label += rest[::max(len(rest) // slots, 1)][:slots]
    to_label = to_label[:max_labels]

    for nd in to_label:
        a = ang_of[nd]
        deg = np.degrees(a)
        flip = 90 < (deg % 360) < 270          # keep text upright on the left half
        ax.text(1.035 * np.cos(a), 1.035 * np.sin(a), nd,
                ha='right' if flip else 'left', va='center',
                rotation=deg + 180 if flip else deg, rotation_mode='anchor',
                fontsize=7, color='#0b0b0b', zorder=5)

    ax.legend(handles=[
        mpatches.Patch(color=OUT_COLOR,
                       label=f'u/{ego} replied to alter  ({G.out_degree(ego)})'),
        mpatches.Patch(color=IN_COLOR,
                       label=f'alter replied to u/{ego}  ({G.in_degree(ego)})'),
    ], loc='upper right', bbox_to_anchor=(1.0, 1.0), frameon=False, fontsize=8.5)

    fig.text(0.02, 0.975, title or f'Ego network of u/{ego}',
             fontsize=13, color='#0b0b0b', va='top', ha='left')
    fig.text(0.02, 0.945, subtitle or (
        f'{len(alters)} alters · {G.number_of_edges()} directed edges · '
        + ('all edge weights tied at 1.0, so width is uniform' if tied
           else 'edge width = interaction weight')
        + f' · {len(to_label)} alters labelled'),
        fontsize=8.5, color=MUTED, va='top', ha='left')

    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.42, 1.42)
    ax.set_aspect('equal'); ax.axis('off')
    fig.subplots_adjust(left=0.02, right=0.98, top=0.92, bottom=0.02)
    return fig, ax


fig, ax = draw_ego_network(G, EGO)
fig.savefig(os.path.join(FIG_DIR, f'ego_network_{EGO}.png'), dpi=140, facecolor=SURFACE)
plt.show()

### 9.1 The graph without submission-level edges

When most incoming edges are top-level comments on the ego's submissions, the picture is
dominated by one-off passers-by. Restricting to comment-to-comment replies shows the
conversational core — the accounts the ego actually exchanged comments with.

In [ ]:
# The same network using only comment-to-comment replies (replies to submissions excluded), with a figure.
conv = interactions[interactions['kind'] == 'comment_reply']
print(f'{len(conv)} comment-to-comment interactions '
      f'({len(interactions) - len(conv)} submission-level excluded)')

if len(conv):
    conv_edges = compute_edge_weights(conv)
    Gc = nx.DiGraph()
    Gc.add_node(EGO, role='ego')
    for _, e in conv_edges.iterrows():
        Gc.add_edge(e['source'], e['target'], weight=float(e['weight']),
                    n_interactions=int(e['n_interactions']),
                    n_posts=int(e['n_posts']), direction=e['direction'])
    print(f'nodes {Gc.number_of_nodes()}  edges {Gc.number_of_edges()}')
    display(conv_edges)
    fig, ax = draw_ego_network(
        Gc, EGO, max_labels=Gc.number_of_nodes() - 1, figsize=(9, 9),
        title=f'Ego network of u/{EGO} — comment-to-comment replies only',
        subtitle=f'{Gc.number_of_nodes() - 1} alters · {Gc.number_of_edges()} directed '
                 f'edges · submission-level replies excluded')
    fig.savefig(os.path.join(FIG_DIR, f'ego_network_{EGO}_conversational.png'),
                dpi=140, facecolor=SURFACE)
    plt.show()
else:
    print('no comment-to-comment interactions in this ego network')

## 10. Ranked edge table

In [ ]:
# Top 25 edges by weight, with how much the discount removed and whether the pair is reciprocated.
ranked = edges.copy()
ranked['raw_vs_weighted'] = ranked['n_interactions'] - ranked['weight']
ranked['reciprocated'] = [
    ((r['target'] if r['direction'] == 'out' else r['source']) in mutual)
    for _, r in ranked.iterrows()
]
print('Top 25 edges by weight')
ranked.head(25)

In [ ]:
# One row per alter: weights and counts in each direction, ranked by total weight.
print('Alters ranked by total weight in both directions:')
alter_summary = (
    pd.concat([
        edges[edges['direction'] == 'out'][['target', 'weight', 'n_interactions', 'n_posts']]
            .rename(columns={'target': 'alter', 'weight': 'w_out',
                             'n_interactions': 'n_out', 'n_posts': 'posts_out'}),
        edges[edges['direction'] == 'in'][['source', 'weight', 'n_interactions', 'n_posts']]
            .rename(columns={'source': 'alter', 'weight': 'w_in',
                             'n_interactions': 'n_in', 'n_posts': 'posts_in'}),
    ])
    .groupby('alter').sum(numeric_only=True)
    .assign(total_weight=lambda d: d.filter(like='w_').sum(axis=1))
    .sort_values('total_weight', ascending=False)
)
alter_summary.head(25)

## 11. Undirected graph — weighted by raw interaction count

A second, simpler view of the same data. Direction is collapsed: one undirected edge per
alter, weighted by the **plain number of interactions** in either direction.

This is deliberately *not* the weighting from section 7 — there is no repetition discount
and no `tau`. An edge of weight 5 means five interactions, however they were distributed
over posts and time. The two graphs are compared at the end of the section.

In [ ]:
U = nx.Graph()
U.add_node(EGO, role='ego')

# Collapse the interaction log onto the unordered pair {ego, alter}.
interactions['alter'] = np.where(interactions['source'] == EGO,
                                 interactions['target'], interactions['source'])

undirected = (
    interactions.groupby('alter')
    .agg(weight=('interaction_id', 'size'),            # raw count - no discount
         n_posts=('link_id', 'nunique'),
         n_ego_to_alter=('direction', lambda s: int((s == 'out').sum())),
         n_alter_to_ego=('direction', lambda s: int((s == 'in').sum())),
         first_seen=('created_utc', 'min'),
         last_seen=('created_utc', 'max'),
         subreddits=('subreddit', lambda s: ','.join(sorted(set(s)))))
    .reset_index()
    .sort_values(['weight', 'alter'], ascending=[False, True])
    .reset_index(drop=True)
)

for _, e in undirected.iterrows():
    U.add_edge(EGO, e['alter'],
               weight=int(e['weight']),
               n_posts=int(e['n_posts']),
               n_ego_to_alter=int(e['n_ego_to_alter']),
               n_alter_to_ego=int(e['n_alter_to_ego']))

print(f'undirected graph: {U.number_of_nodes()} nodes, {U.number_of_edges()} edges')
print(f'weight range: {undirected["weight"].min()} - {undirected["weight"].max()}')
print(f'total weight : {undirected["weight"].sum()}')
undirected.head(15)

### 11.1 Checks

The undirected graph must be a pure collapse of the interaction log — nothing added, nothing
lost, and no interaction counted twice.

In [ ]:
ucheck = []
def ucheck_(name, condition, detail=''):
    ucheck.append((name, bool(condition)))
    print(f'  [{"PASS" if condition else "FAIL"}] {name}{"  " + detail if detail else ""}')

# --- conservation ---
ucheck_('edge weights sum to the total interaction count',
        undirected['weight'].sum() == len(interactions),
        f'{undirected["weight"].sum()} vs {len(interactions)}')
ucheck_('every alter appears exactly once',
        undirected['alter'].is_unique and len(undirected) == len(alters),
        f'{len(undirected)} rows, {len(alters)} alters')
ucheck_('node set matches the directed graph', set(U.nodes()) == set(G.nodes()))
ucheck_('each weight is the sum of its two directed counts',
        (undirected['weight']
         == undirected['n_ego_to_alter'] + undirected['n_alter_to_ego']).all())

# --- it really is undirected and still a star on the ego ---
ucheck_('graph is undirected', not U.is_directed())
ucheck_('every edge is incident to the ego', all(EGO in (u, v) for u, v in U.edges()))
ucheck_('no self-loop on the ego', not U.has_edge(EGO, EGO))
ucheck_('one edge per alter', U.number_of_edges() == len(alters))
ucheck_('ego degree equals the alter count', U.degree(EGO) == len(alters))

# --- weights are raw counts, not the discounted ones ---
ucheck_('all weights are positive integers',
        undirected['weight'].dtype.kind in 'iu' and (undirected['weight'] >= 1).all())
merged = undirected.merge(
    edges.assign(alter=np.where(edges['source'] == EGO, edges['target'], edges['source']))
         .groupby('alter')
         .agg(directed_weight=('weight', 'sum'),
              directed_interactions=('n_interactions', 'sum')).reset_index(),
    on='alter')
ucheck_('raw count matches the directed interaction totals',
        (merged['weight'] == merged['directed_interactions']).all())
ucheck_('raw count is never below the discounted weight',
        (merged['weight'] >= merged['directed_weight'] - 1e-9).all())

print()
print('=' * 68)
print('ALL UNDIRECTED CHECKS PASSED' if all(c for _, c in ucheck)
      else f'FAILURES: {[n for n, c in ucheck if not c]}')
assert all(c for _, c in ucheck)

### 11.2 How the two weightings differ

Where an alter is met on several posts, or repeatedly within one, the raw count and the
discounted weight come apart. This lists the alters where they differ most.

In [ ]:
# Raw interaction count versus discounted weight per alter.
cmp = merged[['alter', 'weight', 'directed_weight', 'n_posts',
              'n_ego_to_alter', 'n_alter_to_ego']].copy()
cmp = cmp.rename(columns={'weight': 'raw_count', 'directed_weight': 'discounted'})
cmp['difference'] = cmp['raw_count'] - cmp['discounted']
cmp = cmp.sort_values('difference', ascending=False).reset_index(drop=True)

if cmp['difference'].max() < 1e-9:
    print('The two weightings are identical on this data: every alter interacted with the')
    print('ego exactly once, so there is nothing for the repetition discount to reduce.')
    print(f'All {len(cmp)} edges have raw_count = discounted = 1.')
else:
    print(f'{int((cmp["difference"] > 1e-9).sum())} alter(s) weighted differently by the '
          f'two schemes:')
print()
cmp.head(15)

### 11.3 Plot

In [ ]:
# Figure: radial undirected ego network, edge width = number of interactions.
def draw_undirected_ego(U, ego, max_labels=16, figsize=(12, 12),
                        title=None, subtitle=None):
    """Radial plot of an undirected ego network. One series (interaction count), so
    width carries weight and no legend box is needed - the subtitle names the encoding."""
    alters_u = [n for n in U.nodes() if n != ego]
    wt = {n: U[ego][n]['weight'] for n in alters_u}
    ordered = sorted(alters_u, key=lambda n: (-wt[n], n))

    n = len(ordered)
    two_rings = n > 60
    pos = {ego: np.array([0.0, 0.0])}
    ang_of = {}
    for i, node in enumerate(ordered):
        a = 2 * np.pi * i / max(n, 1) - np.pi / 2
        r = 1.0 if (not two_rings or i % 2 == 0) else 0.78
        ang_of[node] = a
        pos[node] = np.array([r * np.cos(a), r * np.sin(a)])

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_facecolor(SURFACE); fig.patch.set_facecolor(SURFACE)

    wmin, wmax = min(wt.values()), max(wt.values())
    tied = (wmax - wmin) == 0
    for nd in ordered:
        frac = 0.0 if tied else (wt[nd] - wmin) / (wmax - wmin)
        ax.plot([0, pos[nd][0]], [0, pos[nd][1]],
                color=IN_COLOR, lw=1.0 if tied else 0.6 + 3.4 * frac,
                alpha=0.45 if tied else 0.35 + 0.5 * frac,
                solid_capstyle='round', zorder=2)

    sizes = [34 if tied else 34 + 130 * ((wt[nd] - wmin) / max(wmax - wmin, 1e-9))
             for nd in ordered]
    ax.scatter([pos[nd][0] for nd in ordered], [pos[nd][1] for nd in ordered],
               s=sizes, c='#ffffff', edgecolors=MUTED, linewidths=0.8, zorder=3)
    ax.scatter([0], [0], s=420, c=EGO_COLOR, zorder=4)
    ax.text(0, -0.075, f'u/{ego}', ha='center', va='top', color=EGO_COLOR,
            fontsize=9, fontweight='bold', zorder=5,
            bbox=dict(boxstyle='round,pad=0.22', fc=SURFACE, ec='none', alpha=0.9))

    # heaviest edges first; when weights tie, spread the labels evenly round the ring
    if tied:
        step = max(len(ordered) // max_labels, 1)
        to_label = ordered[::step][:max_labels]
    else:
        to_label = ordered[:max_labels]

    for nd in to_label:
        a = ang_of[nd]
        deg = np.degrees(a)
        flip = 90 < (deg % 360) < 270
        lab = nd if tied else f'{nd} ({wt[nd]})'
        ax.text(1.035 * np.cos(a), 1.035 * np.sin(a), lab,
                ha='right' if flip else 'left', va='center',
                rotation=deg + 180 if flip else deg, rotation_mode='anchor',
                fontsize=7, color='#0b0b0b', zorder=5)

    fig.text(0.02, 0.975, title or f'Ego network of u/{ego} — undirected',
             fontsize=13, color='#0b0b0b', va='top', ha='left')
    fig.text(0.02, 0.945, subtitle or (
        f'{len(alters_u)} alters · {U.number_of_edges()} undirected edges · '
        + (f'every alter interacted exactly {wmin}×, so width is uniform' if tied
           else 'edge width = number of interactions')
        + f' · {len(to_label)} alters labelled'),
        fontsize=8.5, color=MUTED, va='top', ha='left')

    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.42, 1.42)
    ax.set_aspect('equal'); ax.axis('off')
    fig.subplots_adjust(left=0.02, right=0.98, top=0.92, bottom=0.02)
    return fig, ax


fig, ax = draw_undirected_ego(U, EGO)
fig.savefig(os.path.join(FIG_DIR, f'ego_network_{EGO}_undirected.png'),
            dpi=140, facecolor=SURFACE)
plt.show()

### 11.4 Degree distribution

With one edge per alter the informative distribution is over *edge weights* — how many
interactions each alter accounts for.

In [ ]:
# Distribution of interactions per alter in the undirected view.
vc = undirected['weight'].value_counts().sort_index()
print('interactions per alter:')
for k, v in vc.items():
    print(f'  {k:>3} interaction(s): {v:>4} alter(s)  {"#" * min(v, 60)}')
print()
print(f'ego weighted degree: {U.degree(EGO, weight="weight")} '
      f'(= {len(interactions)} interactions)')
print(f'ego degree         : {U.degree(EGO)} (= {len(alters)} alters)')
print(f'mean interactions per alter: {undirected["weight"].mean():.3f}')

## 12. Save outputs

In [ ]:
# edge lists + interaction log
edges.to_parquet(os.path.join(OUT_DIR, f'ego_edges_{EGO}.parquet'),
                 engine='pyarrow', compression='zstd')
undirected.to_parquet(os.path.join(OUT_DIR, f'ego_edges_{EGO}_undirected.parquet'),
                      engine='pyarrow', compression='zstd')

inter_out = interactions.drop(columns=['datetime'])
inter_out.to_parquet(os.path.join(OUT_DIR, f'ego_interactions_{EGO}.parquet'),
                     engine='pyarrow', compression='zstd')

# graphs for gephi / igraph / later notebooks
nx.write_graphml(G, os.path.join(OUT_DIR, f'ego_network_{EGO}.graphml'))
nx.write_graphml(U, os.path.join(OUT_DIR, f'ego_network_{EGO}_undirected.graphml'))

# read back and confirm the round trip
rb_edges = pd.read_parquet(os.path.join(OUT_DIR, f'ego_edges_{EGO}.parquet'))
rb_undir = pd.read_parquet(os.path.join(OUT_DIR, f'ego_edges_{EGO}_undirected.parquet'))
rb_graph = nx.read_graphml(os.path.join(OUT_DIR, f'ego_network_{EGO}.graphml'))
rb_ugraph = nx.read_graphml(os.path.join(OUT_DIR, f'ego_network_{EGO}_undirected.graphml'))
assert rb_edges.shape == edges.shape
assert np.allclose(rb_edges['weight'].sort_values(), edges['weight'].sort_values())
assert rb_graph.number_of_edges() == G.number_of_edges()
assert rb_graph.number_of_nodes() == G.number_of_nodes()
assert rb_undir.shape == undirected.shape
assert (rb_undir['weight'].sort_values().values
        == undirected['weight'].sort_values().values).all()
assert not rb_ugraph.is_directed(), 'undirected graph did not round trip as undirected'
assert rb_ugraph.number_of_edges() == U.number_of_edges()
assert rb_ugraph.number_of_nodes() == U.number_of_nodes()

print('PASS  round trip verified (directed + undirected)')
for d in (OUT_DIR, FIG_DIR):
    for f in sorted(os.listdir(d)):
        if f.startswith('ego_'):
            print(f'  {os.path.join(d, f)}  '
                  f'({os.path.getsize(os.path.join(d, f)) / 1024:.1f} KB)')

## 13. Summary

In [ ]:
# Summary of the ego network and its checks.
print(f'Ego network — u/{EGO}')
print('=' * 68)
print(f'alters                      : {len(alters)}')
print(f'directed edges              : {G.number_of_edges()}')
print(f'  ego -> alter              : {G.out_degree(EGO)}')
print(f'  alter -> ego              : {G.in_degree(EGO)}')
print(f'reciprocated alters         : {len(mutual)}')
print(f'interactions behind them    : {len(interactions)}')
print(f'distinct posts involved     : {interactions["link_id"].nunique()}')
print(f'subreddits                  : {sorted(interactions["subreddit"].unique())}')
print(f'window                      : {interactions["datetime"].min()} -> '
      f'{interactions["datetime"].max()}')
print(f'tau                         : {TAU_SECONDS}s ({TAU_SECONDS/3600:.0f}h)')
print(f'weight range                : {edges["weight"].min():.3f} - '
      f'{edges["weight"].max():.3f}')
print()

n_repeat_pairs = int((edges['n_interactions'] > edges['n_posts']).sum())
n_multipost    = int((edges['n_posts'] > 1).sum())
print(f'edges spanning >1 post      : {n_multipost}')
print(f'edges with within-post repeats: {n_repeat_pairs}')
if n_repeat_pairs == 0 and n_multipost == 0:
    print()
    print('NOTE  every alter interacted with the ego exactly once, on one post, so every')
    print('      edge weight is 1.0. The repetition discount is implemented and verified')
    print('      (section 7.1) but has nothing to discount on this account: it is small')
    print('      and short-lived, with no recurring contacts and no reciprocated pairs.')
    print('      Weight here is not a discriminating variable - the structure is.')
print()
print()
print('Undirected view (raw interaction counts, no discount):')
print(f'  edges                     : {U.number_of_edges()}')
print(f'  total weight              : {undirected["weight"].sum()} '
      f'(= the interaction count)')
print(f'  weight range              : {undirected["weight"].min()} - '
      f'{undirected["weight"].max()}')
print()
print('All property tests (7.1), data checks (7.2), undirected checks (11.1)')
print('and the I/O round trip (12) passed.')